# Transformers und Kontext – Konzepte

**Geschäftliches Problem:** Die Wortreihenfolge und den Kontext berücksichtigen, bevor einem Stellenanzeigen-Portal Kategorien vorgeschlagen werden.

Ein Modell unterstützt eine Entscheidung; es trifft die geschäftliche Entscheidung nicht für uns.

## Lernziele

- Token-Positionen und eine einzelne Attention-Aktualisierung erklären.
- Zwischen BERT-Merkmalen (Features) und Kategorievorhersagen unterscheiden.
- Das Training mit maskierten Tokens (Masked-Token-Training) und die Generierung des nächsten Tokens (Next-Token-Generation) gegenüberstellen.

## Datensatz und praktischer Umfang

Die Token-/Vektorbeispiele und die Wahrscheinlichkeiten bei der Dekodierung in diesem Konzept-Notebook sind künstlich erstellt. Hier wird kein Transformer trainiert.

Die zugehörige [Übung]\(transformers_exercise.ipynb) und [Lösung]\(transformers_solution.ipynb) verwenden die **5'011 echten, anonymisierten Stellenanzeigen**, die der Seite 37 aus Woche 08 zugrunde liegen, und keine künstlich erzeugten Stellenanzeigen. Sie vergleichen Worthäufigkeiten, TF-IDF und ein vortrainiertes, eingefrorenes mehrsprachiges BERT-Modell anhand der ursprünglichen chronologischen Datenaufteilungen.

Der gesamte Code ist hier bereits vorgegeben. Führe die Codezellen von oben nach unten aus.

## Bibliotheken und Einstellungen

Verwende den vorbereiteten Python-3.11-Kernel im Codespace. Es wird weder eine GPU noch ein API-Schlüssel benötigt.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 7
np.random.seed(SEED)
DATA_ROOT = Path('../../../00_Data')

## 1. Ausgangsmodell (Baseline): Ein statischer Mittelwert verliert die Wortreihenfolge

Dieselbe Menge an Wörtern ergibt unabhängig von ihrer Reihenfolge denselben statischen Mittelwert. Kontextabhängige Modelle können die Kombinationen aus Tokens und ihren Positionen berücksichtigen, bevor die Vektoren zusammengefasst werden (Pooling).

In [2]:
first = ['engineer', 'supports', 'customers']
second = ['customers', 'supports', 'engineer']
display(pd.DataFrame({'position': [1, 2, 3], 'sentence A': first, 'sentence B': second}))

,position,sentence A,sentence B
0,1,engineer,customers
1,2,supports,supports
2,3,customers,engineer


## 2. Eine einzelne Self-Attention-Aktualisierung

Halte eine Token-Position fest. Die berechneten Scores für die anderen Positionen werden durch die Softmax-Funktion in Gewichte umgewandelt. Multipliziere jedes Gewicht mit dem zugehörigen Value-Vektor und addiere die Ergebnisse.

Jeder Attention-Head besitzt eigene gelernte Transformationen. Die Attention-Gewichte werden für jede Eingabe neu berechnet. Sie werden weder als eine einzige dauerhafte Tabelle gespeichert noch können sie als kausale Erklärungen verwendet werden.

In [3]:
scores = np.array([2., 1., 0.])
weights = np.exp(scores - scores.max())
weights /= weights.sum()
values = np.array([[1., 0.], [0., 1.], [1., 1.]])
print('Constructed weights:', weights.round(3))
print('New vector for the selected position:', (weights @ values).round(3))

Constructed weights: [0.665 0.245 0.09 ]
New vector for the selected position: [0.755 0.335]


## 3. BERT: Kontextabhängige Merkmale vor einem neuen Klassifikator

BERT steht für *Bidirectional Encoder Representations from Transformers* (bidirektionale Encoder-Repräsentationen aus Transformern).

Beim Masked Language Modelling werden bestimmte Tokens verdeckt. Das Modell versucht, die ursprünglichen Tokens vorherzusagen, indem es den Kontext auf beiden Seiten berücksichtigt. Die richtigen Zielwörter stammen dabei aus dem Text selbst.

In der Übung bleibt BERT eingefroren, das heisst, seine bereits gelernten Parameter werden nicht verändert. Ein neuer Klassifikator lernt anhand der zusammengefassten Merkmale (Pooling), Stellenanzeigen den entsprechenden Kategorien zuzuordnen.

In [4]:
token_features = np.array([[1., 0., 2.], [0., 2., 1.], [2., 1., 0.]])
pooled = token_features.mean(axis=0)
print('Constructed token matrix shape:', token_features.shape)
print('Pooled feature shape:', pooled.shape, '| Values:', pooled)

Constructed token matrix shape: (3, 3)
Pooled feature shape: (3,) | Values: [1. 1. 1.]


## 4. Encoder versus Decoder

Ein Encoder kann den Eingabekontext auf beiden Seiten eines Tokens berücksichtigen.

Ein Encoder-Decoder-Modell kann zusätzlich Text generieren und dabei auf den vom Encoder verarbeiteten Ausgangstext zugreifen.

Ein Decoder-only-Modell im GPT-Stil verwendet einen kausalen Kontext. Das bedeutet, dass es nur die vorhergehenden Tokens berücksichtigt und das nächste Token vorhersagt.

Beim Training werden die bereits bekannten nächsten Tokens als Zielwerte verwendet. Bei der Textgenerierung wird das ausgewählte Token an den bisherigen Text angehängt. Anschliessend wird der Vorgang wiederholt.

In [5]:
prefix = ['We', 'need', 'a', 'software']
vocabulary = ['engineer', 'analyst', '.']
next_probabilities = np.array([0.7, 0.2, 0.1])
chosen = vocabulary[int(np.argmax(next_probabilities))]
print('Constructed greedy continuation:', ' '.join(prefix + [chosen]))

Constructed greedy continuation: We need a software engineer


## Evaluation und Interpretation

Diese Berechnungen überprüfen einen Mechanismus, nicht das Sprachverständnis.

Vergleiche in der Übung den tatsächlichen Klassifikator, der eingefrorene Merkmale verwendet, mit den Baseline-Modellen auf Basis von Worthäufigkeiten und TF-IDF.

Eine flüssige Textfortsetzung bedeutet nicht automatisch, dass der generierte Inhalt sachlich korrekt ist.

## Zusammenfassung und geschäftliche Auswirkungen

Ein Merkmalsvektor (Feature Vector) ist keine Wahrscheinlichkeit für eine bestimmte Kategorie. Pooling verändert die Form der Ausgabe, nicht das Trainingsziel. Auch eingefrorene Modellparameter können weiterhin Vektoren erzeugen, die von der jeweiligen Eingabe abhängen.

## Einschränkungen

Dies ist ein kleines Lehrbeispiel und kein Nachweis dafür, wie gut das Modell im praktischen Einsatz funktioniert. Halte die Evaluation getrennt vom Training und von der Modellauswahl. Untersuche die Fehler und berücksichtige die Entscheidung des Nutzers, statt dich nur auf eine einzelne Leistungskennzahl zu konzentrieren.

## Weiterführende Übungen

- Erkläre, warum Attention allein noch keinen vollständigen Transformer ausmacht.
- Führt das Hinzufügen von Pooling dazu, dass BERT für die Informationssuche (Retrieval) trainiert ist?
- Warum werden die Modellgewichte durch Prompting normalerweise nicht aktualisiert?

## References

**Required reading:** BERT overview: https://huggingface.co/docs/transformers/model_doc/bert
**Instructor references:** Vaswani et al. (2017), *Attention Is All You Need*, https://arxiv.org/abs/1706.03762;
Devlin et al. (2019), *BERT*, https://aclanthology.org/N19-1423/;
Brown et al. (2020), *Language Models are Few-Shot Learners*, https://arxiv.org/abs/2005.14165.

### Jupyter notebook - footer info

Include your name, submission date and a short description of your changes when submitting.